In [1]:
import sys; sys.path.append('..')  # shared utils.py lives in the repo root
import torch
import numpy as np
import pandas as pd
import pyro
from pyro import poutine
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
from dmm import DMM
from utils import log_returns

c:\Users\mikah\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# load SPY and compute log-returns
spy = pd.read_csv('../data/spy_ohlcv.csv', index_col=0, parse_dates=True, skiprows = [1, 2])
ret = log_returns(spy['Close']).dropna().values.astype(np.float32)

# train/test split
K          = 15
n_train    = 5000
n_test     = 250
train_ret  = ret[:n_train]
test_ret   = ret[n_train:n_train + n_test]
print(f"train: {train_ret.shape}  test: {test_ret.shape}")

train: (5000,)  test: (250,)


In [6]:
# split training returns into non-overlapping sequences of length T
T = 20  # ~1 trading month per sequence
n_sequences = len(train_ret) // T
train_data = torch.tensor(train_ret[:n_sequences * T].reshape(n_sequences, T, 1))
print(f"train dataset: {train_data.shape}  ({n_sequences} sequences of length {T})")

# test set as a single sequence (infer handles the windowing)
test_data = torch.tensor(test_ret).unsqueeze(0).unsqueeze(-1)  # (1, n_test, 1)
print(f"test dataset:  {test_data.shape}")

train dataset: torch.Size([250, 20, 1])  (250 sequences of length 20)
test dataset:  torch.Size([1, 250, 1])


In [7]:
# dataloader
batch_size = 32
loader = DataLoader(TensorDataset(train_data), batch_size=batch_size, shuffle=True)
print(f"batches per epoch: {len(loader)}")

batches per epoch: 8


In [8]:
# model
x_dim          = 1
z_dim          = 8
hidden_dim     = 64
rnn_hidden_dim = 64
rnn_num_layers = 1

dmm = DMM(x_dim, z_dim, hidden_dim, rnn_hidden_dim, rnn_num_layers)

In [9]:
def elbo_terms(model, guide, x):
    # returns (reconstruction, kl) as scalars
    guide_trace = poutine.trace(guide).get_trace(x)
    model_trace = poutine.trace(poutine.replay(model, trace=guide_trace)).get_trace(x)

    guide_trace.compute_log_prob()
    model_trace.compute_log_prob()
    
    reconstruction = sum(
        site["log_prob_sum"]
        for site in model_trace.nodes.values()
        if site["type"] == "sample" and site["is_observed"]
    )
    kl = sum(
        guide_trace.nodes[name]["log_prob_sum"] - model_trace.nodes[name]["log_prob_sum"]
        for name, site in guide_trace.nodes.items()
        if site["type"] == "sample"
    )
    return reconstruction.item(), kl.item()

In [10]:
# training
pyro.clear_param_store()
optimizer = Adam({"lr": 1e-3})
svi = SVI(dmm.model, dmm.guide, optimizer, loss=Trace_ELBO())

num_epochs = 200
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for (batch,) in loader:
        epoch_loss += svi.step(batch)
    if epoch % 50 == 0:
        with torch.no_grad():
            recon, kl = elbo_terms(dmm.model, dmm.guide, train_data)
        print(f"epoch {epoch:4d}  loss: {epoch_loss / len(loader):10.4f}  "
              f"recon: {recon:10.4f}  kl: {kl:10.4f}")

epoch    0  loss:  1096.0365  recon: -2375.4951  kl:  4321.2656
epoch   50  loss: -1868.4950  recon: 15158.9902  kl:   172.3160
epoch  100  loss: -1905.3008  recon: 15171.8330  kl:   103.6322
epoch  150  loss: -1888.0826  recon: 15367.0000  kl:   172.6765


In [11]:
from scipy.stats import t as scipy_t

# Gaussian baseline
baseline_mu    = torch.tensor(train_ret.mean())
baseline_sigma = torch.tensor(train_ret.std())
nll_baseline_gauss = -torch.distributions.Normal(baseline_mu, baseline_sigma).log_prob(test_data).sum() / (test_data.shape[1] - K)

# Student-t baseline (MLE fit via scipy)
df_fit, mu_fit, scale_fit = scipy_t.fit(train_ret)
nll_baseline_t = -torch.distributions.StudentT(
    torch.tensor(df_fit), torch.tensor(mu_fit), torch.tensor(scale_fit)
).log_prob(test_data).sum() / (test_data.shape[1] - K)

print(f"NLL baseline (Gaussian):   {nll_baseline_gauss.item():.4f}")
print(f"NLL baseline (Student-t):  {nll_baseline_t.item():.4f}  (df={df_fit:.2f})")


NLL baseline (Gaussian):   -3.5184
NLL baseline (Student-t):  -3.6851  (df=2.81)


In [12]:
# test set evaluation
with torch.no_grad():
    nll_direct     = dmm.predict_nll(test_data, K=K, method='direct')
    nll_transition = dmm.predict_nll(test_data, K=K, method='transition')

print(f"NLL baseline Gaussian:  {nll_baseline_gauss.item():.4f}")
print(f"NLL baseline Student-t: {nll_baseline_t.item():.4f}")
print(f"NLL direct:             {nll_direct:.4f}")
print(f"NLL transition:         {nll_transition:.4f}")

NLL baseline Gaussian:  -3.5184
NLL baseline Student-t: -3.6851
NLL direct:             -3.4136
NLL transition:         -3.4124


In [13]:
train_data_seq = torch.tensor(train_ret).unsqueeze(0).unsqueeze(-1)  # (1, 6000, 1)

with torch.no_grad():
    nll_train_direct     = dmm.predict_nll(train_data_seq, K=K, method='direct')
    nll_train_transition = dmm.predict_nll(train_data_seq, K=K, method='transition')

baseline_mu_train    = torch.tensor(train_ret.mean())
baseline_sigma_train = torch.tensor(train_ret.std())
nll_train_baseline_gauss = -torch.distributions.Normal(baseline_mu_train, baseline_sigma_train).log_prob(train_data_seq).sum() / (train_data_seq.shape[1] - K)

df_fit_train, mu_fit_train, scale_fit_train = scipy_t.fit(train_ret)
nll_train_baseline_t = -torch.distributions.StudentT(
    torch.tensor(df_fit_train), torch.tensor(mu_fit_train), torch.tensor(scale_fit_train)
).log_prob(train_data_seq).sum() / (train_data_seq.shape[1] - K)

print(f"Train NLL baseline Gaussian:  {nll_train_baseline_gauss.item():.4f}")
print(f"Train NLL baseline Student-t: {nll_train_baseline_t.item():.4f}  (df={df_fit_train:.2f})")
print(f"Train NLL direct:             {nll_train_direct:.4f}")
print(f"Train NLL transition:         {nll_train_transition:.4f}")


Train NLL baseline Gaussian:  -2.9806
Train NLL baseline Student-t: -3.1063  (df=2.81)
Train NLL direct:             -3.0426
Train NLL transition:         -3.0426


In [15]:
z_means, _ = dmm.infer(test_data, K)
df, x_mu, x_sigma = dmm.emitter(z_means[:, :, :].reshape(-1, dmm.z_dim))
print(f"Mean predicted df:    {df.mean().item():.4f}")
print(f"Mean predicted sigma: {x_sigma.mean().item():.6f}")
print(f"Training sample std:  {train_data.std().item():.6f}")


Mean predicted df:    3.4678
Mean predicted sigma: 0.008146
Training sample std:  0.012395


In [17]:
z_means, _ = dmm.infer(test_data, K)
sigmas = []
with torch.no_grad():
    for t in range(z_means.shape[1]):
        _, _, sigma = dmm.emitter(z_means[:, t, :])
        sigmas.append(sigma.mean().item())

sigmas = np.array(sigmas)
print(f"Sigma std over time: {sigmas.std():.6f}")
print(f"Sigma min: {sigmas.min():.6f}, max: {sigmas.max():.6f}")

Sigma std over time: 0.000044
Sigma min: 0.008124, max: 0.008801
